# MedCLIP-SAMv2 Text+Boxes — Sheffield Dataset — GPU Optimised (from Aug_12)

Same pipeline as `gpu_optimized_lambda_medclipsamv2_textboxes_sheffield.ipynb`,
but `dcm_files` is filtered to start at `Aug_12`. Run this on a **second**
Lambda instance in parallel with the original notebook (which keeps working
through the lower-numbered volumes) to split the Sheffield set across two
GPUs. Both notebooks write `.npz` files keyed by volume index into the same
output directory name, so downloads from both instances merge cleanly into
one local results folder.

Optimisations (same as the base notebook):

| Change | Original | Optimised |
|---|---|---|
| SAM encoder | batch_size=1 per slice | auto-batched (EMBED_BATCH, free-VRAM scaled, backs off on OOM) |
| BiomedCLIP saliency | BATCH=64 (hardcoded) | SAL_BATCH (configurable) |
| Slices | all slices encoded/decoded | only slices containing a target muscle |
| CPU threads | default | `torch.set_num_threads(os.cpu_count())` |

`EMBED_BATCH` is auto-detected from *free* VRAM at startup (other processes
may share the GPU) and adapts down on OOM during the encoder loop.

⚠️ **Requires MuscleMap WB Sheffield segmentations** — run
`lambda_musclemap_wb_sheffield.ipynb` first and download results to
`eval_notebooks/muscle_map_wb/sheffield_segs/` before uploading here.

Data: `~/sheffeld/20440164/Aug_N.dcm`
MM WB segs: `~/musclemap_wb_sheffield_segs/Aug_N_dseg.nii.gz`
Output: `~/medclipsamv2_textboxes_sheffield_segs/Aug_N_mcsam2textboxes.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb/sheffield_segs/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/musclemap_wb_sheffield_segs/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/tmp/docker-desktop-root/run/desktop/mnt/host/c/Users/docto/AppData/Local/Dafne-imaging/Dafne/models/medsam_vit_b.pth" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsam_vit_b.pth
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medclipsamv2_textboxes_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2textboxes/sheffield_segs/
```
**Terminate the instance when done.**

In [1]:
import os, subprocess, sys

# Reduce CUDA allocator fragmentation — must be set before the first CUDA
# context is created (i.e. before torch.cuda is touched anywhere in this process).
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Reinstall PyTorch with a wheel compiled against NumPy 2.x
# (the system torch is built for NumPy 1.x, which conflicts with opencv>=4.10)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'SimpleITK', 'scikit-image', 'pydicom', 'opencv-python-headless'])

import importlib, torch
importlib.invalidate_caches()
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/lib/python3/dist-packages/ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "/usr/lib/python3/dist-packages/traitlets/config/application.py", line 846, in launch_instance
    app.start()
  File "/usr/lib/python3/dist-packages/ipykernel/kernelapp.py", line 677, in start
    s

PyTorch: 2.7.0
CUDA   : True NVIDIA A10


In [2]:
REPO_DIR = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_DIR = os.path.expanduser('~/mcsam2_env')
VENV_PY  = os.path.join(VENV_DIR, 'bin', 'python')

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/HealthX-Lab/MedCLIP-SAMv2.git', REPO_DIR])

if not os.path.exists(VENV_PY):
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])

def venv_pip(*args):
    subprocess.check_call([VENV_PY, '-m', 'pip'] + list(args))

venv_pip('install', '-q', '--upgrade', 'pip')
venv_pip('install', '-q', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124')
venv_pip('install', '-q', '-e', os.path.join(REPO_DIR, 'segment-anything'))
venv_pip('install', '-q', 'git+https://github.com/lucasb-eyer/pydensecrf.git')
# No opencv-python here — it requires numpy>=2 and would break open_clip/grad-cam.
# The batch saliency script uses PIL for image I/O instead.
venv_pip('install', '-q',
    'open_clip_torch', 'SimpleITK', 'Pillow',
    'huggingface_hub', 'transformers<4.46',
    'matplotlib', 'grad-cam', 'pandas', 'tqdm', 'scipy', 'scikit-learn')
# Pin numpy<2 LAST so no earlier install can upgrade it
venv_pip('install', '-q', 'numpy<2')
import subprocess as _sp
_np_ver = _sp.check_output([VENV_PY, '-c', 'import numpy; print(numpy.__version__)'], text=True).strip()
print(f'Venv numpy: {_np_ver}')
assert _np_ver.startswith('1.'), f'Expected numpy 1.x in venv, got {_np_ver}'
print('Venv dependencies installed.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


Venv numpy: 1.26.4
Venv dependencies installed.


In [3]:
import glob, re, shutil, tempfile
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
import cv2
import pydicom
from PIL import Image
from skimage import transform
from segment_anything import sam_model_registry

IMG_DIR    = os.path.expanduser('~/sheffeld/20440164')
MM_SEG_DIR = os.path.expanduser('~/musclemap_wb_sheffield_segs')
OUTPUT_DIR = os.path.expanduser('~/medclipsamv2_textboxes_sheffield_segs')
MEDSAM_CKPT= os.path.expanduser('~/medsam_vit_b.pth')
REPO_DIR   = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_PY    = os.path.expanduser('~/mcsam2_env/bin/python')
VENV_DIR   = os.path.expanduser('~/mcsam2_env')
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
SAM_DEVICE = DEVICE

torch.set_num_threads(os.cpu_count())

os.makedirs(OUTPUT_DIR, exist_ok=True)

for label, path in [('MedSAM ckpt', MEDSAM_CKPT), ('MM WB segs', MM_SEG_DIR)]:
    ok = os.path.exists(path)
    print(f'  {"OK" if ok else "MISSING"}: {label}')
    if not ok:
        raise FileNotFoundError(f'{label} not found — upload it first')

# Only process Aug_<N> volumes with N >= START_FROM — lets a second instance
# work through the back half of the dataset in parallel with another
# notebook/instance that's covering the front half.
START_FROM = 19

dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm'))
     if '_segmentations' not in f
     and int(re.search(r'Aug_(\d+)\.dcm', f).group(1)) >= START_FROM],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
)

# ── GPU batch sizes ───────────────────────────────────────────────────────────
def _auto_embed_batch(device):
    """SAM ViT-B's global-attention blocks scale memory with batch x heads x
    tokens^2 at 1024x1024 (4096 tokens) — far more than a flat per-image cost.
    Use *free* VRAM (not total), since other processes may share the GPU, and
    a conservative budget; the embedding loop also backs off on OOM."""
    if device == 'cpu':
        return 1
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    return max(1, min(16, int((free_gb - 1) / 3.0)))

EMBED_BATCH = _auto_embed_batch(SAM_DEVICE)  # SAM image encoder slices per forward pass
SAL_BATCH   = 128                            # BiomedCLIP saliency slices per forward pass

print(f'START_FROM  : Aug_{START_FROM}')
print(f'{len(dcm_files)} DICOM volumes (>= Aug_{START_FROM})')
print(f'SAM device  : {SAM_DEVICE}')
print(f'CPU threads : {torch.get_num_threads()}')
print(f'Free VRAM   : {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB' if SAM_DEVICE == 'cuda' else '')
print(f'EMBED_BATCH : {EMBED_BATCH}  (SAM encoder slices per forward pass, adapts on OOM)')
print(f'SAL_BATCH   : {SAL_BATCH}  (BiomedCLIP slices per forward pass)')

  OK: MedSAM ckpt
  OK: MM WB segs
START_FROM  : Aug_19
51 DICOM volumes (>= Aug_19)
SAM device  : cuda
CPU threads : 30
Free VRAM   : 21.9 GB
EMBED_BATCH : 6  (SAM encoder slices per forward pass, adapts on OOM)
SAL_BATCH   : 128  (BiomedCLIP slices per forward pass)


In [4]:
# (output_key, text_prompt, mm_wb_label)
# MuscleMap WB uses 7xxx label scheme
MUSCLES = [
    ('L_vastus_lateralis',    'vastus lateralis muscle left thigh MRI axial cross section',   7101),
    ('R_vastus_lateralis',    'vastus lateralis muscle right thigh MRI axial cross section',  7102),
    ('L_vastus_intermedius',  'vastus intermedius muscle left thigh MRI axial cross section', 7111),
    ('R_vastus_intermedius',  'vastus intermedius muscle right thigh MRI axial cross section',7112),
    ('L_vastus_medialis',     'vastus medialis muscle left thigh MRI axial cross section',    7121),
    ('R_vastus_medialis',     'vastus medialis muscle right thigh MRI axial cross section',   7122),
    ('L_rectus_femoris',      'rectus femoris muscle left thigh MRI axial cross section',     7131),
    ('R_rectus_femoris',      'rectus femoris muscle right thigh MRI axial cross section',    7132),
    ('L_sartorius',           'sartorius muscle left thigh MRI axial cross section',          7141),
    ('R_sartorius',           'sartorius muscle right thigh MRI axial cross section',         7142),
    ('L_gracilis',            'gracilis muscle left thigh MRI axial cross section',           7151),
    ('R_gracilis',            'gracilis muscle right thigh MRI axial cross section',          7152),
    ('L_semimembranosus',     'semimembranosus muscle left thigh MRI axial cross section',    7161),
    ('R_semimembranosus',     'semimembranosus muscle right thigh MRI axial cross section',   7162),
    ('L_semitendinosus',      'semitendinosus muscle left thigh MRI axial cross section',     7171),
    ('R_semitendinosus',      'semitendinosus muscle right thigh MRI axial cross section',    7172),
    ('L_biceps_femoris_long', 'biceps femoris long head muscle left thigh MRI axial cross section',  7181),
    ('R_biceps_femoris_long', 'biceps femoris long head muscle right thigh MRI axial cross section', 7182),
    ('L_biceps_femoris_short','biceps femoris short head muscle left thigh MRI axial cross section', 7191),
    ('R_biceps_femoris_short','biceps femoris short head muscle right thigh MRI axial cross section',7192),
    ('L_adductor_magnus',     'adductor magnus muscle left thigh MRI axial cross section',    7201),
    ('R_adductor_magnus',     'adductor magnus muscle right thigh MRI axial cross section',   7202),
    ('L_adductor_longus',     'adductor longus muscle left thigh MRI axial cross section',    7211),
    ('R_adductor_longus',     'adductor longus muscle right thigh MRI axial cross section',   7212),
    ('L_adductor_brevis',     'adductor brevis muscle left thigh MRI axial cross section',    7221),
    ('R_adductor_brevis',     'adductor brevis muscle right thigh MRI axial cross section',   7222),
]
print(f'{len(MUSCLES)} muscles')

26 muscles


In [5]:
import glob as _glob, json as _json
_venv_site = _glob.glob(os.path.join(VENV_DIR, 'lib', 'python3.*', 'site-packages'))
VENV_SITE  = _venv_site[0] if _venv_site else ''

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH']       = VENV_SITE + ':' + SUBPROCESS_ENV.get('PYTHONPATH', '')
SUBPROCESS_ENV['PYTHONNOUSERSITE'] = '1'
SUBPROCESS_ENV['MPLBACKEND']       = 'Agg'

BATCH_SAL_SCRIPT = os.path.expanduser('~/batch_saliency.py')

_SAL_SCRIPT_BODY = '''#!/usr/bin/env python3
"""Batch BiomedCLIP saliency — one model load per volume, GPU-batched inference.
Images are kept on CPU and moved to GPU per-batch to minimise VRAM footprint."""
import argparse, json, os
import numpy as np, torch
from PIL import Image
import open_clip

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument(\'--png-dir\',      required=True)
    p.add_argument(\'--out-base\',     required=True)
    p.add_argument(\'--prompts-json\', required=True)
    p.add_argument(\'--device\',       default=\'cuda\')
    p.add_argument(\'--batch-size\',   type=int, default=64)
    return p.parse_args()

def main():
    args   = parse_args()
    device = args.device if torch.cuda.is_available() else \'cpu\'
    torch.set_num_threads(os.cpu_count())

    with open(args.prompts_json) as f:
        prompts = json.load(f)

    png_files = sorted(
        [x for x in os.listdir(args.png_dir) if x.endswith(\'.png\')],
        key=lambda x: int(x.split(\'.\')[0])
    )
    N = len(png_files)
    print(f\'Loading BiomedCLIP on {device} ...\', flush=True)
    model, _, preprocess = open_clip.create_model_and_transforms(
        \'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224\'
    )
    tokenizer = open_clip.get_tokenizer(
        \'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224\'
    )
    model = model.to(device).eval()

    # Auto-detect safe batch size from free VRAM after model load.
    # ViT-B/16 needs ~6 MB per image (activations + gradients during backward).
    BATCH = args.batch_size
    if str(device) != \'cpu\' and torch.cuda.is_available():
        _free_gb = torch.cuda.mem_get_info()[0] / 1e9
        BATCH    = max(4, min(args.batch_size, int(_free_gb / 0.006)))
        print(f\'[sal] free VRAM after model load: {_free_gb:.1f} GB  SAL_BATCH={BATCH}\', flush=True)

    print(f\'Model ready. {len(prompts)} prompts x {N} slices. batch={BATCH}\', flush=True)

    imgs_pil       = [Image.open(os.path.join(args.png_dir, f)).convert(\'RGB\') for f in png_files]
    W_orig, H_orig = imgs_pil[0].size
    # Keep preprocessed images on CPU — moved to GPU per-batch to avoid VRAM pressure
    img_batch_cpu  = torch.stack([preprocess(im) for im in imgs_pil])

    for name, prompt in prompts.items():
        out_dir = os.path.join(args.out_base, name)
        os.makedirs(out_dir, exist_ok=True)
        print(f\'  [{name}]\', flush=True)

        text_tok = tokenizer([prompt]).to(device)
        with torch.no_grad():
            text_feat = model.encode_text(text_tok)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

        sal_chunks = []
        for b0 in range(0, N, BATCH):
            b1   = min(b0 + BATCH, N)
            imgs = img_batch_cpu[b0:b1].to(device).detach().requires_grad_(True)
            img_feat = model.encode_image(imgs)
            img_feat = img_feat / (img_feat.norm(dim=-1, keepdim=True) + 1e-8)
            sim = (img_feat * text_feat).sum(dim=-1).sum()
            sim.backward()
            sal = imgs.grad.detach().abs().max(dim=1)[0].cpu().numpy()
            sal_chunks.append(sal)
            del imgs
            torch.cuda.empty_cache()

        sal_all = np.concatenate(sal_chunks, axis=0)
        s_min, s_max = sal_all.min(), sal_all.max()
        if s_max > s_min:
            sal_all = (sal_all - s_min) / (s_max - s_min)

        for i, fn in enumerate(png_files):
            sal_u8 = (sal_all[i] * 255).astype(np.uint8)
            Image.fromarray(sal_u8).resize(
                (W_orig, H_orig), Image.BILINEAR
            ).save(os.path.join(out_dir, fn))

    print(\'Batch saliency complete.\', flush=True)

if __name__ == \'__main__\':
    main()
'''

with open(BATCH_SAL_SCRIPT, 'w') as _f:
    _f.write(_SAL_SCRIPT_BODY)
print(f'Batch saliency script written to {BATCH_SAL_SCRIPT}')


# ── numpy <-> torch bridges that survive has_numpy=False ─────────────────────

def _np_to_t(arr: np.ndarray) -> torch.Tensor:
    a = np.ascontiguousarray(arr, dtype=np.float32)
    return torch.frombuffer(a, dtype=torch.float32).clone().reshape(a.shape)

def _t_to_np(t: torch.Tensor) -> np.ndarray:
    t_con = t.detach().cpu().float().clone().contiguous()
    raw   = bytes(t_con.untyped_storage())
    return np.frombuffer(raw, dtype=np.float32).reshape(t_con.shape).copy()


# ── Helpers ───────────────────────────────────────────────────────────────────

def preprocess_slice(sl_arr: np.ndarray) -> torch.Tensor:
    """(H, W) float32 → (3, 1024, 1024) float32 CPU tensor ready for SAM encoder."""
    img_norm = (sl_arr * 255.0 / (sl_arr.max() + 1e-8)).astype(np.float32)
    img_3c   = np.stack([img_norm, img_norm, img_norm], axis=-1)
    img_1024 = cv2.resize(img_3c, (1024, 1024), interpolation=cv2.INTER_CUBIC)
    lo, hi   = img_1024.min(), img_1024.max()
    img_1024 = ((img_1024 - lo) / max(hi - lo, 1e-8)).astype(np.float32)
    arr      = img_1024.transpose(2, 0, 1).copy()
    return torch.frombuffer(arr, dtype=torch.float32).clone().reshape(3, 1024, 1024)


def export_slices_as_png(img_array, out_dir, slice_indices=None):
    os.makedirs(out_dir, exist_ok=True)
    indices = range(img_array.shape[0]) if slice_indices is None else slice_indices
    for i, sl_idx in enumerate(indices):
        sl = img_array[sl_idx]
        sl_norm = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        Image.fromarray(
            np.stack([(sl_norm * 255).astype(np.uint8)] * 3, axis=-1)
        ).save(os.path.join(out_dir, f'{i}.png'))


def run_all_saliencies(png_dir, out_base, muscles, sal_batch):
    """One subprocess: load BiomedCLIP once, run all muscle prompts."""
    prompts = {name: prompt for name, prompt, _ in muscles}
    prompts_path = os.path.join(out_base, '_prompts.json')
    os.makedirs(out_base, exist_ok=True)
    with open(prompts_path, 'w') as f:
        _json.dump(prompts, f)
    sal_env = {**SUBPROCESS_ENV, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
    r = subprocess.run(
        [VENV_PY, BATCH_SAL_SCRIPT,
         '--png-dir',      png_dir,
         '--out-base',     out_base,
         '--prompts-json', prompts_path,
         '--device',       DEVICE,
         '--batch-size',   str(sal_batch)],
        capture_output=True, text=True, env=sal_env,
    )
    if r.returncode != 0:
        raise RuntimeError(f'Batch saliency failed:\n{r.stderr[-3000:]}')
    print(r.stdout[-800:])


def load_saliency_volume(sal_dir, num_slices, target_size=256):
    vol = np.zeros((num_slices, target_size, target_size), dtype=np.float32)
    for i in range(num_slices):
        p = os.path.join(sal_dir, f'{i}.png')
        if os.path.exists(p):
            s = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            if s is not None:
                s = cv2.resize(s, (target_size, target_size),
                               interpolation=cv2.INTER_LINEAR).astype(np.float32)
                vol[i] = (s / 255.0) * 12.0 - 6.0
    return _np_to_t(vol[:, None, :, :])


def get_mm_boxes(seg_array, mm_label, H, W, margin=5, slice_indices=None):
    indices = range(seg_array.shape[0]) if slice_indices is None else slice_indices
    boxes = []
    for sl in indices:
        mask = (seg_array[sl] == mm_label).astype(np.uint8)
        if not mask.any():
            boxes.append(None); continue
        rows = np.where(np.any(mask, axis=1))[0]
        cols = np.where(np.any(mask, axis=0))[0]
        r0, r1 = rows[[0, -1]]; c0, c1 = cols[[0, -1]]
        mH, mW = mask.shape
        box = np.array([
            max(0, c0-margin), max(0, r0-margin),
            min(mW-1, c1+margin), min(mH-1, r1+margin),
        ], dtype=float)
        boxes.append(box / np.array([W, H, W, H]) * 1024)
    return boxes


def medsam_infer(sam_model, img_embed, box_1024, saliency_256, H, W, device):
    box_t = torch.as_tensor(box_1024, dtype=torch.float, device=device)[None, None, :]
    sal_t = saliency_256.to(device)
    with torch.no_grad():
        sparse_emb, dense_emb = sam_model.prompt_encoder(
            points=None, boxes=box_t, masks=sal_t)
        logits, _ = sam_model.mask_decoder(
            image_embeddings=img_embed.to(device),
            image_pe=sam_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False,
        )
    pred = F.interpolate(torch.sigmoid(logits), size=(H, W),
                         mode='bilinear', align_corners=False)
    return (_t_to_np(pred.squeeze()) > 0.5).astype(np.uint8)


print('Helpers defined.')

Batch saliency script written to /home/ubuntu/batch_saliency.py
Helpers defined.


In [6]:
sam_model = sam_model_registry['vit_b'](checkpoint=MEDSAM_CKPT)
sam_model.to(device=SAM_DEVICE).eval()
print('MedSAM loaded on', SAM_DEVICE)

MedSAM loaded on cuda


In [ ]:
import time

for dcm_path in dcm_files:
    idx      = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    mm_path  = os.path.join(MM_SEG_DIR, f'Aug_{idx}_dseg.nii.gz')
    out_path = os.path.join(OUTPUT_DIR, f'Aug_{idx}_mcsam2textboxes.npz')

    if not os.path.exists(mm_path):
        print(f'[skip] no MM WB seg for Aug_{idx}')
        continue
    if os.path.exists(out_path):
        print(f'Skipping (done): Aug_{idx}')
        continue

    print(f'\n═══ Aug_{idx} ═══')
    ds        = pydicom.dcmread(dcm_path)
    img_array = ds.pixel_array.astype(np.float32)
    D, H, W   = img_array.shape

    seg_sitk  = sitk.ReadImage(mm_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk).astype(np.int32)
    print(f'  Image: {img_array.shape}  MM seg: {seg_array.shape}')

    # ── Skip slices that contain none of the 26 target muscles ─────────────
    mm_labels         = np.array([lbl for _, _, lbl in MUSCLES])
    slice_has_muscle  = np.isin(seg_array, mm_labels).any(axis=(1, 2))
    valid_slices      = np.where(slice_has_muscle)[0]
    Dv                = len(valid_slices)
    print(f'  {Dv}/{D} slices contain at least one target muscle')

    all_masks = {name: np.zeros((D, H, W), dtype=np.uint8) for name, _, _ in MUSCLES}

    if Dv == 0:
        np.savez_compressed(out_path, **all_masks)
        print(f'  No target muscles found — saved empty masks → {out_path}')
        continue

    tmp_root  = tempfile.mkdtemp(prefix='mctbs_')

    try:
        png_dir = os.path.join(tmp_root, 'slices')
        export_slices_as_png(img_array, png_dir, slice_indices=valid_slices)

        # ── Batched SAM encoder (valid slices only) ───────────────────────────
        # Adaptive: SAM's global-attention blocks make memory use depend on
        # batch size super-linearly, so back off and retry on OOM rather than
        # failing the whole volume.
        print(f'  Computing SAM embeddings (EMBED_BATCH={EMBED_BATCH} initial)...')
        embeddings  = []
        cur_batch   = EMBED_BATCH
        b0 = 0
        while b0 < Dv:
            b1           = min(b0 + cur_batch, Dv)
            batch_slices = valid_slices[b0:b1]
            batch = torch.stack([preprocess_slice(img_array[sl]) for sl in batch_slices])
            batch = batch.to(SAM_DEVICE)
            try:
                with torch.no_grad():
                    embs = sam_model.image_encoder(batch)   # (B, 256, 64, 64)
            except torch.cuda.OutOfMemoryError:
                del batch
                torch.cuda.empty_cache()
                if cur_batch == 1:
                    raise
                new_batch = max(1, cur_batch // 2)
                print(f'    OOM at batch={cur_batch}, retrying with batch={new_batch}')
                cur_batch = new_batch
                continue
            embeddings.extend([embs[i:i+1].cpu() for i in range(embs.shape[0])])
            del batch, embs
            b0 = b1
        torch.cuda.empty_cache()
        print(f'  {Dv} embeddings ready')

        # ── BiomedCLIP saliency (valid slices only) ───────────────────────────
        sal_base = os.path.join(tmp_root, 'saliencies')
        print(f'  Running batch saliency (SAL_BATCH={SAL_BATCH})...')
        run_all_saliencies(png_dir, sal_base, MUSCLES, SAL_BATCH)

        # ── Pre-load all saliency volumes and boxes ───────────────────────────
        # Single I/O phase before the decoder — eliminates per-muscle disk gaps.
        print(f'  Pre-loading {len(MUSCLES)} saliency volumes...')
        sal_vols   = []
        boxes_list = []
        for muscle_name, _, mm_label in MUSCLES:
            sal_dir = os.path.join(sal_base, muscle_name)
            sal_vols.append(load_saliency_volume(sal_dir, Dv))                                       # (Dv, 1, 256, 256)
            boxes_list.append(get_mm_boxes(seg_array, mm_label, H, W, slice_indices=valid_slices))   # Dv items

        # ── Per-slice decoder: all muscles in one forward pass ────────────────
        # SAM supports N prompts for one image in a single call.
        # Looping over slices (not muscles) batches all 26 muscle prompts per
        # slice → Dv calls instead of 26×Dv (26× fewer decoder launches), and
        # only over slices that actually contain a target muscle.
        print(f'  Decoding masks ({Dv} slices × {len(MUSCLES)} muscles/call)...')
        _decode_t0     = time.time()
        PROGRESS_EVERY = 50

        for v_idx in range(Dv):
            sl_idx    = valid_slices[v_idx]
            valid_m   = [(m, boxes_list[m][v_idx]) for m in range(len(MUSCLES))
                         if boxes_list[m][v_idx] is not None]
            invalid_m = [m for m in range(len(MUSCLES))
                         if boxes_list[m][v_idx] is None]

            # No-box slices: threshold saliency directly (no GPU needed)
            for m in invalid_m:
                sal_np = _t_to_np(sal_vols[m][v_idx, 0])
                all_masks[MUSCLES[m][0]][sl_idx] = cv2.resize(
                    (sal_np > 0).astype(np.uint8), (W, H),
                    interpolation=cv2.INTER_NEAREST)

            if valid_m:
                m_idxs  = [v[0] for v in valid_m]
                N       = len(valid_m)

                # Stack all N muscle prompts for this slice
                boxes_t = torch.tensor(
                    [v[1] for v in valid_m], dtype=torch.float, device=SAM_DEVICE
                ).unsqueeze(1)                                            # (N, 1, 4)
                sals_t  = torch.cat(
                    [sal_vols[m][v_idx:v_idx+1] for m in m_idxs]
                ).to(SAM_DEVICE)                                          # (N, 1, 256, 256)

                with torch.no_grad():
                    sparse_emb, dense_emb = sam_model.prompt_encoder(
                        points=None, boxes=boxes_t, masks=sals_t)
                    # image_embeddings=(1,...): SAM repeats it N times internally
                    logits, _ = sam_model.mask_decoder(
                        image_embeddings=embeddings[v_idx].to(SAM_DEVICE),
                        image_pe=sam_model.prompt_encoder.get_dense_pe(),
                        sparse_prompt_embeddings=sparse_emb,   # (N, 2, 256)
                        dense_prompt_embeddings=dense_emb,     # (N, 256, 64, 64)
                        multimask_output=False,
                    )
                # logits: (N, 1, 64, 64) → upsample → threshold
                pred     = F.interpolate(torch.sigmoid(logits), size=(H, W),
                                         mode='bilinear', align_corners=False)
                masks_np = (_t_to_np(pred.squeeze(1)) > 0.5).astype(np.uint8)  # (N, H, W)

                for j, m in enumerate(m_idxs):
                    all_masks[MUSCLES[m][0]][sl_idx] = masks_np[j]

            if (v_idx + 1) % PROGRESS_EVERY == 0 or v_idx + 1 == Dv:
                elapsed = time.time() - _decode_t0
                rate    = (v_idx + 1) / elapsed if elapsed > 0 else 0
                eta     = (Dv - (v_idx + 1)) / rate if rate > 0 else 0
                print(f'    {v_idx+1}/{Dv} slices decoded'
                      f'  ({rate:.1f} slices/s, ETA {eta:.0f}s)')

        for name, _, _ in MUSCLES:
            print(f'  [{name}] {int(all_masks[name].sum()):,} voxels')

        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved → {out_path}')

    except Exception as exc:
        import traceback
        print(f'  ERROR on Aug_{idx}: {exc}')
        traceback.print_exc()

    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)

print('\nAll done.')


═══ Aug_19 ═══
  Image: (1071, 320, 256)  MM seg: (1071, 320, 256)
  427/1071 slices contain at least one target muscle
  Computing SAM embeddings (EMBED_BATCH=6 initial)...


In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Output files: {len(results)} / {len(dcm_files)}')
if results:
    s = np.load(results[0])
    for k in sorted(s.files):
        print(f'  {k}: voxels={int(s[k].sum()):,}')